# Custom policy

Smoke test on a small full-set demo. `TinyFullSetPolicy` in `src/aRieL/agents/policies/tiny_linear_policy.py` is the template. The released architecture is `FullSetISABPolicy` in `full_set_isab_policy.py`.

Needs the `train` extra.

In [ ]:
from dataclasses import replace

from aRieL import make_demo_targets
from aRieL.utils.config import ActionConfig, FullSetActionConfig, load_preset

targets = make_demo_targets(8)
cfg = load_preset("demo", reward="default")
cfg = replace(
    cfg,
    action=ActionConfig(
        type="full_set",
        full_set=FullSetActionConfig(k_filter=8, n_max=8),
    ),
)

In [ ]:
from sb3_contrib import MaskablePPO

from aRieL.agents import make_training_envs
from aRieL.agents.policies.tiny_linear_policy import TinyFullSetPolicy

train_env = make_training_envs(cfg, n_envs=1, seed=0, targets=targets)
model = MaskablePPO(
    TinyFullSetPolicy,
    train_env,
    n_steps=64,
    batch_size=64,
    verbose=0,
)
model.learn(total_timesteps=256)
train_env.close()

In [ ]:
from aRieL import ArielEnv

env = ArielEnv(config=cfg, targets=targets)
obs, info = env.reset(seed=0)
for _ in range(3):
    action, _ = model.predict(
        obs, action_masks=info["action_mask"], deterministic=True
    )
    obs, reward, terminated, truncated, info = env.step(int(action))
    print(int(action), round(float(reward), 3))
    if terminated or truncated:
        break
env.close()